<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 RBench Reproduction with Cosmos Framework

This notebook walks through generating the [RBench](https://huggingface.co/datasets/DAGroup-PKU/RBench) robotics video-generation benchmark with a Cosmos3 checkpoint (Cosmos3-Nano by default, or Cosmos3-Super) using the native Cosmos Framework PyTorch entrypoint:

```bash
python -m cosmos_framework.scripts.inference
```

RBench is an **Image-to-Video (I2V)** benchmark: each case provides a single condition image and a prompt, and the model generates a short clip.

- **650 cases** split across 9 category files (e.g. `common_manipulation`, `dual_arm`, `humanoid`, ...).
- Each case is conditioned on one image (`imgs/<image_path>` in the dataset) plus a detailed `json_upsampled_prompt` and a shared `negative_prompt`.
- Generation target: **120 generated frames at 24 FPS** (we run `num_frames=121`, where frame 0 is the conditioning image).

We walk through one demo case (`common_manipulation_0001`) end-to-end. The full 650-case run is one cell at the end.

## Scoring Overview and Reference Scores

The official RBench **overall score** is the equal-weight mean of nine indicators: five task-category scores from the 250-sample task split and four embodiment scores from the 400-sample embodiment split. It is not a sample-weighted mean over all 650 videos.

```text
RBench overall = mean(5 task-category scores + 4 embodiment scores)
```

Each task-category score is the mean normalized VLM score, `(raw_score - 1) / 4`, for its 50 samples. For each embodiment, ReVidgen computes `Task Completion (TC)` and penalized `Visual Quality (VQ)` per sample; its indicator is `TS = (TC + VQ) / 2`, averaged over the embodiment's 100 samples. Sections 11 and 12 compute these two complementary splits, and Section 13 combines them into the official overall score.

The full-set reference numbers below are the published RBench Image-to-Video overall scores.

| Variant | RBench overall |
| --- | ---: |
| Cosmos3-Super | 58.1% |
| Cosmos3-Nano | 58.4% |

## Prerequisites

- Linux machine with NVIDIA GPU access (default recipe uses 8 GPUs).
- Model access on Hugging Face. Either run `uvx hf@latest auth login` or set `HF_TOKEN` in the environment.
- `uv >= 0.11.3` installed (https://docs.astral.sh/uv/getting-started/installation/).
- `git-lfs` on PATH. The RBench dataset (~22 GB) is downloaded by cloning the Hugging Face dataset repo, which stores images and checkpoints with Git LFS.
- Cache/output paths with enough disk space.



## 1. Configure Paths and Environment

All paths default to sensible locations under this `cosmos` checkout. Override any of them by exporting before launching the notebook:

```bash
export COSMOS3_REPO=/path/to/cosmos-framework
export COSMOS3_UV_GROUP=cu130-train   # or cu128-train
export UV_PROJECT_ENVIRONMENT=/path/to/large/uv/venvs/cosmos3-rbench
export COSMOS3_NUM_GPUS=8
export HF_HOME=/path/to/large/huggingface/cache
export CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7
export RBENCH_MODEL_VARIANT=Nano   # or Super (which Cosmos3 checkpoint to evaluate)
export RBENCH_DATASET_ROOT=/path/to/RBench
export RBENCH_OUTPUT_ROOT=/path/to/rbench/outputs
```

In [ ]:
from pathlib import Path
import os
import socket


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


def free_local_port() -> str:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return str(sock.getsockname()[1])


def default_framework_repo(root: Path) -> Path:
    for candidate in (root / "packages" / "cosmos-framework", root / "packages" / "cosmos3"):
        if (candidate / "pyproject.toml").exists() and (candidate / "cosmos_framework").exists():
            return candidate
    return root / "packages" / "cosmos-framework"


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", default_framework_repo(COSMOS_ROOT))).resolve()
COSMOS3_GIT_URL = os.environ.get("COSMOS3_GIT_URL", "git@github.com:NVIDIA/cosmos-framework.git")
COSMOS3_UV_GROUP = os.environ.get("COSMOS3_UV_GROUP", "cu130-train")
COSMOS3_UV_ENV = Path(os.environ.get("UV_PROJECT_ENVIRONMENT", COSMOS3_REPO / ".venv")).resolve()
COSMOS3_NUM_GPUS = os.environ.get("COSMOS3_NUM_GPUS", "8")
CUDA_VISIBLE_DEVICES = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1,2,3,4,5,6,7")

RBENCH_NOTEBOOK_ROOT = COSMOS_ROOT / "evaluation" / "cosmos3" / "generator" / "rbench"
RBENCH_ASSETS = RBENCH_NOTEBOOK_ROOT / "assets"
RBENCH_PROMPTS_DIR = RBENCH_ASSETS / "prompts"
RBENCH_HF_URL = os.environ.get("RBENCH_HF_URL", "https://huggingface.co/datasets/DAGroup-PKU/RBench")
RBENCH_DATASET_ROOT = Path(
    os.environ.get("RBENCH_DATASET_ROOT", RBENCH_NOTEBOOK_ROOT / "RBench")
).resolve()
RBENCH_IMGS_DIR = RBENCH_DATASET_ROOT / "imgs"
RBENCH_OUTPUT_ROOT = Path(
    os.environ.get("RBENCH_OUTPUT_ROOT", RBENCH_NOTEBOOK_ROOT / "outputs")
).resolve()
RBENCH_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEMO_CASE_ID = os.environ.get("RBENCH_DEMO_CASE", "common_manipulation_0001")
# Model variant to evaluate: "Super" or "Nano".
MODEL_VARIANT = os.environ.get("RBENCH_MODEL_VARIANT", "Nano")
assert MODEL_VARIANT in ("Super", "Nano"), f"RBENCH_MODEL_VARIANT must be 'Super' or 'Nano', got {MODEL_VARIANT!r}"
CHECKPOINT = os.environ.get("RBENCH_CHECKPOINT", f"Cosmos3-{MODEL_VARIANT}")

MASTER_ADDR = os.environ.get("COSMOS3_MASTER_ADDR", "127.0.0.1")
MASTER_PORT_I2V = os.environ.get("COSMOS3_I2V_MASTER_PORT", free_local_port())

for key, value in [
    ("COSMOS_ROOT", COSMOS_ROOT),
    ("COSMOS3_REPO", COSMOS3_REPO),
    ("COSMOS3_UV_ENV", COSMOS3_UV_ENV),
    ("COSMOS3_NUM_GPUS", COSMOS3_NUM_GPUS),
    ("CUDA_VISIBLE_DEVICES", CUDA_VISIBLE_DEVICES),
    ("RBENCH_PROMPTS_DIR", RBENCH_PROMPTS_DIR),
    ("RBENCH_DATASET_ROOT", RBENCH_DATASET_ROOT),
    ("RBENCH_OUTPUT_ROOT", RBENCH_OUTPUT_ROOT),
    ("DEMO_CASE_ID", DEMO_CASE_ID),
    ("MODEL_VARIANT", MODEL_VARIANT),
    ("CHECKPOINT", CHECKPOINT),
]:
    print(f"{key}={value}")

# Export for the %%bash cells below.
os.environ["RBENCH_NOTEBOOK_ROOT"] = str(RBENCH_NOTEBOOK_ROOT)
os.environ["COSMOS3_REPO"] = str(COSMOS3_REPO)
os.environ["COSMOS3_GIT_URL"] = COSMOS3_GIT_URL
os.environ["COSMOS3_UV_GROUP"] = COSMOS3_UV_GROUP
os.environ["COSMOS3_UV_ENV"] = str(COSMOS3_UV_ENV)
os.environ["UV_PROJECT_ENVIRONMENT"] = str(COSMOS3_UV_ENV)
os.environ["COSMOS3_NUM_GPUS"] = COSMOS3_NUM_GPUS
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
os.environ["RBENCH_HF_URL"] = RBENCH_HF_URL
os.environ["RBENCH_DATASET_ROOT"] = str(RBENCH_DATASET_ROOT)
os.environ["RBENCH_OUTPUT_ROOT"] = str(RBENCH_OUTPUT_ROOT)
os.environ["DEMO_CASE_ID"] = DEMO_CASE_ID
os.environ["CHECKPOINT"] = CHECKPOINT
os.environ["COSMOS3_MASTER_ADDR"] = MASTER_ADDR
os.environ["COSMOS3_I2V_MASTER_PORT"] = MASTER_PORT_I2V

## 2. Clone or Reuse Cosmos Framework

In [ ]:
%%bash
set -euo pipefail

mkdir -p "$(dirname "$COSMOS3_REPO")"

if [ -f "$COSMOS3_REPO/pyproject.toml" ] && [ -d "$COSMOS3_REPO/cosmos_framework" ]; then
  echo "Using existing framework checkout: $COSMOS3_REPO"
elif [ -e "$COSMOS3_REPO" ]; then
  echo "COSMOS3_REPO exists but is not a Cosmos Framework checkout: $COSMOS3_REPO"
  exit 1
else
  echo "Cloning $COSMOS3_GIT_URL into $COSMOS3_REPO"
  git clone "$COSMOS3_GIT_URL" "$COSMOS3_REPO"
fi

cd "$COSMOS3_REPO"
git status --short --branch
git remote -v

## 3. Install Native PyTorch Dependencies

Installs framework dependencies with the requested CUDA group (default `cu130-train`).

In [ ]:
%%bash
set -euo pipefail

if ! command -v uv >/dev/null 2>&1; then
  echo "uv is not installed. Install it first: https://docs.astral.sh/uv/getting-started/installation/"
  exit 1
fi

export GIT_LFS_SKIP_SMUDGE=1
cd "$COSMOS3_REPO"
export UV_PROJECT_ENVIRONMENT="${UV_PROJECT_ENVIRONMENT:-$COSMOS3_UV_ENV}"
echo "Using UV_PROJECT_ENVIRONMENT=$UV_PROJECT_ENVIRONMENT"
uv sync --all-extras --group="$COSMOS3_UV_GROUP"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "uv sync completed, but expected Python is missing: $COSMOS3_UV_ENV/bin/python"
  exit 1
fi

## 4. Verify GPU and Python Environment

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "Missing $COSMOS3_UV_ENV/bin/python"
  echo "Run the Install Native PyTorch Dependencies cell first."
  exit 1
fi
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" "$COSMOS3_UV_ENV/bin/python" - <<'PY'
import torch
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(f"device {index}:", torch.cuda.get_device_name(index))
PY

## 5. Download the RBench Dataset

RBench is hosted on Hugging Face at [`DAGroup-PKU/RBench`](https://huggingface.co/datasets/DAGroup-PKU/RBench). We download it by cloning the dataset repo with Git LFS.

Layout (under `$RBENCH_DATASET_ROOT`):

```
RBench/
├── assets/        # benchmark assets
├── checkpoints/   # BERT checkpoints (for the official RBench scorer; not used here)
├── imgs/          # 650 condition images, organized by category (imgs/<category>/<name>.jpg)
└── prompts/       # 9 source prompt files
```

> The positive/negative prompts used for generation come from the **local** assets at `assets/prompts/` (which include `json_upsampled_prompt` and `negative_prompt`). Only the condition images are read from the cloned dataset.

In [ ]:
%%bash
set -euo pipefail

if [ -d "$RBENCH_DATASET_ROOT/imgs" ]; then
  echo "RBench dataset already present at $RBENCH_DATASET_ROOT"
  ls "$RBENCH_DATASET_ROOT"
  exit 0
fi

if ! command -v git-lfs >/dev/null 2>&1; then
  echo "git-lfs is required to download the RBench dataset (LFS-backed images/checkpoints)." >&2
  echo "Install it: https://git-lfs.com/  (e.g. apt-get install -y git-lfs)" >&2
  exit 1
fi

git lfs install
mkdir -p "$(dirname "$RBENCH_DATASET_ROOT")"
git clone "$RBENCH_HF_URL" "$RBENCH_DATASET_ROOT"

echo "--- contents of $RBENCH_DATASET_ROOT ---"
ls "$RBENCH_DATASET_ROOT"

## 6. Load Prompts and Preview the Demo Case

We combine the 9 local category prompt files under `assets/prompts/` into a single `name -> entry` mapping (650 cases total). Each entry carries the `json_upsampled_prompt` (used as the positive prompt) and the `negative_prompt`. The condition image for a case is `imgs/<image_path>` inside the cloned dataset.

In [ ]:
import json
from IPython.display import Image, display

prompt_files = sorted(RBENCH_PROMPTS_DIR.glob("*.json"))
assert prompt_files, f"No prompt files found under {RBENCH_PROMPTS_DIR}"

PROMPTS: dict[str, dict] = {}
for path in prompt_files:
    for row in json.loads(path.read_text()):
        name = row["name"]
        if name in PROMPTS:
            raise ValueError(f"Duplicate prompt name across files: {name}")
        PROMPTS[name] = row

print(f"Loaded {len(PROMPTS)} prompts from {len(prompt_files)} files:")
for path in prompt_files:
    print(" ", path.name)
assert len(PROMPTS) == 650, f"expected 650 prompts, got {len(PROMPTS)}"

entry = PROMPTS[DEMO_CASE_ID]
demo_image = RBENCH_IMGS_DIR / entry["image_path"]
print("\ndemo case:", DEMO_CASE_ID)
print("condition image:", demo_image)
print("short prompt:", entry.get("prompt"))

if demo_image.exists():
    display(Image(filename=str(demo_image), width=480))
else:
    print("Condition image not found yet - run the dataset download cell first.")

## 7. Helper Functions

**I2V recipe**

- **Conditioning**: the single condition image. The model loads it as the first generated frame.
- **Output length**: 121 frames at 24 fps. Frame 0 is the conditioning image; frames 1–120 are the 5 seconds of generated content.
- **Sampling**: `num_steps=50`, `guidance=6.0`, `shift=10.0`, `seed=0` (all other sampler settings left at framework defaults).
- **Positive prompt**: the per-case `json_upsampled_prompt` string, passed through verbatim.
- **Negative prompt**: the shared `negative_prompt` string, passed through verbatim.

Helpers:

- `case_image_path(case_id)` — resolve the condition image for a case from the cloned dataset.
- `build_i2v_row(case_id)` — assemble the inference JSONL row for a case.
- `build_i2v_input_jsonl(case_id, dst)` — write a one-line JSONL for a single case.

In [ ]:
I2V_NUM_FRAMES = 121
I2V_FPS = 24
I2V_RESOLUTION = "720"
I2V_ASPECT_RATIO = "16,9"
I2V_NUM_STEPS = 50
I2V_GUIDANCE = 6.0
I2V_SHIFT = 10.0
I2V_SEED = 0


def case_image_path(case_id: str) -> Path:
    entry = PROMPTS[case_id]
    image_path = RBENCH_IMGS_DIR / entry["image_path"]
    if not image_path.exists():
        raise FileNotFoundError(f"condition image not found for {case_id}: {image_path}")
    return image_path


def build_i2v_row(case_id: str) -> dict:
    entry = PROMPTS[case_id]
    return {
        "aspect_ratio": I2V_ASPECT_RATIO,
        "fps": I2V_FPS,
        "guidance": I2V_GUIDANCE,
        "model_mode": "image2video",
        "name": case_id,
        "negative_prompt": entry["negative_prompt"],
        "num_frames": I2V_NUM_FRAMES,
        "num_outputs": 1,
        "num_steps": I2V_NUM_STEPS,
        "prompt": entry["json_upsampled_prompt"],
        "resolution": I2V_RESOLUTION,
        "seed": I2V_SEED,
        "shift": I2V_SHIFT,
        "vision_path": str(case_image_path(case_id)),
    }


def build_i2v_input_jsonl(case_id: str, dst_jsonl: Path) -> Path:
    dst_jsonl.parent.mkdir(parents=True, exist_ok=True)
    dst_jsonl.write_text(json.dumps(build_i2v_row(case_id)) + "\n")
    return dst_jsonl


def display_video(path: Path, width: int = 480) -> None:
    from IPython.display import Video, display
    display(Video(filename=str(path), embed=True, width=width))

## 8. I2V — Build the Input JSONL for the Demo Case

In [ ]:
i2v_run_dir = RBENCH_OUTPUT_ROOT / "i2v" / DEMO_CASE_ID
i2v_run_dir.mkdir(parents=True, exist_ok=True)

i2v_input_jsonl = i2v_run_dir / "input.jsonl"
i2v_output_dir = i2v_run_dir / "raw"
i2v_output_dir.mkdir(parents=True, exist_ok=True)

build_i2v_input_jsonl(DEMO_CASE_ID, i2v_input_jsonl)

os.environ["I2V_INPUT"] = str(i2v_input_jsonl)
os.environ["I2V_OUTPUT_DIR"] = str(i2v_output_dir)

print("I2V_INPUT =", i2v_input_jsonl)
print("I2V_OUTPUT_DIR =", i2v_output_dir)
print()
print("row preview:")
print(i2v_input_jsonl.read_text()[:600], "...")

### Run I2V Inference

We use the **latency** parallelism preset with context-parallel sharding across all visible GPUs (`--cp-size=$COSMOS3_NUM_GPUS`). For a single-case run on 8×H100, this completes in roughly 1–2 minutes.

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_I2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$I2V_INPUT" \
  -o "$I2V_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

### Preview the Raw Output

The raw generation is a 121-frame mp4 at 24 fps (frame 0 is the conditioning image). We keep it as-is.

In [ ]:
raw_outputs = sorted(i2v_output_dir.rglob("vision.mp4"))
if raw_outputs:
    raw_mp4 = raw_outputs[0]
    print("raw output:", raw_mp4)
    display_video(raw_mp4)
else:
    print("No vision.mp4 found yet - run the inference cell first.")

## 9. (Optional) Run All 650 Cases

The cell below mirrors the demo flow but writes a single combined JSONL covering every case (all 9 categories), then the following bash cell generates all 650 I2V outputs in one run.

Run them only if you intend to generate the full benchmark.

In [ ]:
RUN_ALL = False  # set to True to enable the full 650-case sweep

if RUN_ALL:
    case_ids = sorted(PROMPTS.keys())
    assert len(case_ids) == 650

    all_inputs_dir = RBENCH_OUTPUT_ROOT / "i2v_full" / "inputs"
    all_raw_dir = RBENCH_OUTPUT_ROOT / "i2v_full" / "raw"
    all_inputs_dir.mkdir(parents=True, exist_ok=True)
    all_raw_dir.mkdir(parents=True, exist_ok=True)

    all_jsonl = all_inputs_dir / "all_650.jsonl"
    with all_jsonl.open("w") as fp:
        for case_id in case_ids:
            fp.write(json.dumps(build_i2v_row(case_id)) + "\n")

    os.environ["I2V_FULL_INPUT"] = str(all_jsonl)
    os.environ["I2V_FULL_OUTPUT_DIR"] = str(all_raw_dir)
    print("I2V_FULL_INPUT =", all_jsonl)
    print("I2V_FULL_OUTPUT_DIR =", all_raw_dir)
    print(f"Wrote {len(case_ids)} rows. Run the next bash cell to generate all 650 I2V outputs.")
else:
    print("Set RUN_ALL = True above and re-run to enable the full 650-case sweep.")

### Generate All 650 I2V Outputs

In [ ]:
%%bash
set -euo pipefail

if [ -z "${I2V_FULL_INPUT:-}" ]; then
  echo "Set RUN_ALL = True in the previous Python cell and re-run it first."
  exit 0
fi

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_I2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$I2V_FULL_INPUT" \
  -o "$I2V_FULL_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

## 10. Configure and Install RBench Scoring

This common setup serves both RBench evaluation splits while keeping scoring isolated from the Cosmos generation environment:

- `.venv-rbench-scorer`: Transformers/Qwen environment used by the five-task scorer and the VQA portion of four-embodiment scoring.
- `.venv-rbench-ops`: GroundingDINO, SAM2, CoTracker, and Q-Align environment used by four-embodiment motion scoring.

Neither environment is activated in the notebook kernel. Every scorer command uses its explicit interpreter. Enable `RUN_SCORING` for the five-task split and `RUN_EMBODIMENT_SCORING` for the four-embodiment split. A complete official overall score requires both.


### 10.1 Configure Scoring Paths

Override these before launching the notebook if needed:

```bash
export REVIDGEN_REPO=/path/to/ReVidgen
export REVIDGEN_SCORER_ENV=/path/to/.venv-rbench-scorer
export RBENCH_OPS_ENV=/path/to/.venv-rbench-ops
export QWEN_MODEL_PATH=Qwen/Qwen2.5-VL-72B-Instruct
export RBENCH_SCORE_RAW_DIR=/path/to/outputs/i2v_full/raw
export RBENCH_SCORE_HF_HOME=/path/to/huggingface/cache
export RBENCH_OPS_CUDA_VISIBLE_DEVICES=0
```


In [ ]:
RUN_SCORING = False  # RBench 5-tasks (250-sample subset)
RUN_EMBODIMENT_SCORING = False  # RBench 4-embodiment VQA + motion metrics

RBENCH_SCORE_TASKS = [
    "common_manipulation",
    "long-horizon_planning",
    "multi-entity_collaboration",
    "spatial_relationship",
    "visual_reasoning",
]
RBENCH_EMBODIMENTS = ["dual_arm", "humanoid", "single_arm", "quad"]

_checkpoint_slug = CHECKPOINT.lower().replace("-", "_")
MODEL_NAME = os.environ.get("RBENCH_MODEL_NAME", _checkpoint_slug)

REVIDGEN_REPO = Path(
    os.environ.get("REVIDGEN_REPO", RBENCH_NOTEBOOK_ROOT / "scorers" / "ReVidgen")
).resolve()
REVIDGEN_GIT_URL = os.environ.get("REVIDGEN_GIT_URL", "https://github.com/DAGroup-PKU/ReVidgen.git")
REVIDGEN_COMMIT = os.environ.get("REVIDGEN_COMMIT", "b03df27f0376faa148dcd8cd620a1989a32ca979")
REVIDGEN_SCORER_ENV = Path(
    os.environ.get("REVIDGEN_SCORER_ENV", REVIDGEN_REPO / ".venv-rbench-scorer")
).resolve()
RBENCH_OPS_ENV = Path(
    os.environ.get("RBENCH_OPS_ENV", REVIDGEN_REPO / ".venv-rbench-ops")
).resolve()

QWEN_MODEL_PATH = os.environ.get("QWEN_MODEL_PATH", "Qwen/Qwen2.5-VL-72B-Instruct")
RBENCH_SCORE_RAW_DIR = Path(
    os.environ.get("RBENCH_SCORE_RAW_DIR", RBENCH_OUTPUT_ROOT / "i2v_full" / "raw")
).resolve()
RBENCH_SCORE_HF_HOME = Path(
    os.environ.get("RBENCH_SCORE_HF_HOME", REVIDGEN_REPO / ".hf_cache")
).resolve()
RBENCH_OPS_CUDA_VISIBLE_DEVICES = os.environ.get(
    "RBENCH_OPS_CUDA_VISIBLE_DEVICES", CUDA_VISIBLE_DEVICES.split(",")[0]
)

RBENCH_4EMB_VQA_NAME = f"{MODEL_NAME}_4emb_vqa"
RBENCH_4EMB_MOTION_NAME = f"{MODEL_NAME}_4emb_motion"

for key, value in [
    ("RUN_SCORING", RUN_SCORING),
    ("RUN_EMBODIMENT_SCORING", RUN_EMBODIMENT_SCORING),
    ("MODEL_NAME", MODEL_NAME),
    ("REVIDGEN_REPO", REVIDGEN_REPO),
    ("REVIDGEN_SCORER_ENV", REVIDGEN_SCORER_ENV),
    ("RBENCH_OPS_ENV", RBENCH_OPS_ENV),
    ("QWEN_MODEL_PATH", QWEN_MODEL_PATH),
    ("RBENCH_SCORE_RAW_DIR", RBENCH_SCORE_RAW_DIR),
    ("RBENCH_SCORE_HF_HOME", RBENCH_SCORE_HF_HOME),
    ("RBENCH_OPS_CUDA_VISIBLE_DEVICES", RBENCH_OPS_CUDA_VISIBLE_DEVICES),
]:
    print(f"{key}={value}")

os.environ.update({
    "RUN_SCORING": str(RUN_SCORING),
    "RUN_EMBODIMENT_SCORING": str(RUN_EMBODIMENT_SCORING),
    "REVIDGEN_REPO": str(REVIDGEN_REPO),
    "REVIDGEN_GIT_URL": REVIDGEN_GIT_URL,
    "REVIDGEN_COMMIT": REVIDGEN_COMMIT,
    "REVIDGEN_SCORER_ENV": str(REVIDGEN_SCORER_ENV),
    "RBENCH_OPS_ENV": str(RBENCH_OPS_ENV),
    "MODEL_NAME": MODEL_NAME,
    "QWEN_MODEL_PATH": QWEN_MODEL_PATH,
    "RBENCH_SCORE_TASKS": " ".join(RBENCH_SCORE_TASKS),
    "RBENCH_EMBODIMENTS": " ".join(RBENCH_EMBODIMENTS),
    "RBENCH_SCORE_HF_HOME": str(RBENCH_SCORE_HF_HOME),
    "RBENCH_OPS_CUDA_VISIBLE_DEVICES": RBENCH_OPS_CUDA_VISIBLE_DEVICES,
    "RBENCH_4EMB_VQA_NAME": RBENCH_4EMB_VQA_NAME,
    "RBENCH_4EMB_MOTION_NAME": RBENCH_4EMB_MOTION_NAME,
})

if not (RUN_SCORING or RUN_EMBODIMENT_SCORING):
    print("\nEnable one of the scoring flags above before running the installation/preparation cells.")


### 10.2 Install the Isolated Scoring Environments

Run the checked-in setup scripts as subprocesses. Do not activate either environment and do not install scorer packages with the notebook kernel's Python.

- Five-task scoring needs `.venv-rbench-scorer`.
- Four-embodiment scoring needs both `.venv-rbench-scorer` and `.venv-rbench-ops`.

The operator installer also downloads the RBench checkpoints and four embodiment prompt files. It compiles GroundingDINO for the configured CUDA architecture and may take a while on its first run.


In [ ]:
%%bash
set -euo pipefail

if [ "${RUN_SCORING:-False}" != "True" ] && [ "${RUN_EMBODIMENT_SCORING:-False}" != "True" ]; then
  echo "Both scoring flags are False; skipping scorer installation."
  exit 0
fi

: "${RBENCH_NOTEBOOK_ROOT:?Run the main configuration cell before installing the scorers}"
SETUP_DIR="$RBENCH_NOTEBOOK_ROOT"

echo "--- Installing/verifying the Qwen VLM scorer environment ---"
bash "$SETUP_DIR/setup_rbench_scorer.sh"

if [ "${RUN_EMBODIMENT_SCORING:-False}" = "True" ]; then
  echo "--- Installing/verifying the 4-embodiment operator environment ---"
  bash "$SETUP_DIR/setup_rbench_embodiment_scorer.sh"
fi


### 10.3 Verify the Explicit Scorer Interpreters

These checks deliberately invoke the venv interpreters by path. The notebook kernel remains in the Cosmos generation environment.


In [ ]:
%%bash
set -euo pipefail

if [ "${RUN_SCORING:-False}" != "True" ] && [ "${RUN_EMBODIMENT_SCORING:-False}" != "True" ]; then
  echo "Both scoring flags are False; skipping scorer verification."
  exit 0
fi

VLM_PY="$REVIDGEN_SCORER_ENV/bin/python"
test -x "$VLM_PY"
PYTHONNOUSERSITE=1 "$VLM_PY" - <<'PY'
import torch, transformers
print("VLM torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("VLM transformers:", transformers.__version__)
print("VLM CUDA available:", torch.cuda.is_available(), "GPUs:", torch.cuda.device_count())
assert torch.cuda.is_available()
assert int(transformers.__version__.split(".", 1)[0]) == 4
PY

if [ "${RUN_EMBODIMENT_SCORING:-False}" = "True" ]; then
  OPS_PY="$RBENCH_OPS_ENV/bin/python"
  test -x "$OPS_PY"
  REVIDGEN_REPO="$REVIDGEN_REPO" PYTHONNOUSERSITE=1 "$OPS_PY" - <<'PY'
import glob, os, torch
repo = os.environ["REVIDGEN_REPO"]
print("Ops torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("Ops CUDA available:", torch.cuda.is_available())
print("GroundingDINO extensions:", glob.glob(repo + "/pkgs/Grounded-Segment-Anything/GroundingDINO/groundingdino/_C*.so"))
assert torch.cuda.is_available()
assert glob.glob(repo + "/pkgs/Grounded-Segment-Anything/GroundingDINO/groundingdino/_C*.so")
PY
fi


## 11. Score the Five Task Categories

The task-oriented split contains 250 samples: 50 each for Common Manipulation, Long-Horizon Planning, Multi-Entity Collaboration, Spatial Relationship, and Visual Reasoning. Each category indicator is the mean of its normalized VLM scores, `(raw_score - 1) / 4`.

### 11.1 Prepare Scorer Prompts and Organize Generated Videos

The scorer expects `data/<model_name>/<task>/videos/NNNN.mp4` and maps numeric filenames to prompt positions. This cell copies the official scorer prompts and creates symlinks to the generated `vision.mp4` files.


In [ ]:
if not RUN_SCORING:
    print("RUN_SCORING is False - set it to True in the config cell (10.1) and re-run.")
else:
    dataset_prompts_dir = RBENCH_DATASET_ROOT / "prompts"
    scorer_prompts_dir = REVIDGEN_REPO / "data" / "prompts"
    scorer_prompts_dir.mkdir(parents=True, exist_ok=True)

    if not RBENCH_SCORE_RAW_DIR.exists():
        raise FileNotFoundError(
            f"No raw outputs at {RBENCH_SCORE_RAW_DIR}. Generate videos first "
            "(Section 9 full sweep, or set RBENCH_SCORE_RAW_DIR to a demo raw dir)."
        )

    # Map each generated case (dir name) -> its raw vision.mp4.
    raw_by_case: dict[str, Path] = {}
    for mp4 in sorted(RBENCH_SCORE_RAW_DIR.rglob("vision.mp4")):
        case_name = mp4.parent.name.split("__")[0]
        raw_by_case.setdefault(case_name, mp4)
    print(f"Found {len(raw_by_case)} generated clips under {RBENCH_SCORE_RAW_DIR}")

    total_linked = 0
    for task in RBENCH_SCORE_TASKS:
        src_prompt = dataset_prompts_dir / f"{task}_prompts.json"
        if not src_prompt.exists():
            print(f"  [skip] scorer prompt file missing: {src_prompt}")
            continue

        # Copy the scorer prompt file into the scorer's data/prompts/.
        dst_prompt = scorer_prompts_dir / f"{task}_prompts.json"
        dst_prompt.write_text(src_prompt.read_text())

        prompt_rows = json.loads(src_prompt.read_text())
        videos_dir = REVIDGEN_REPO / "data" / MODEL_NAME / task / "videos"
        videos_dir.mkdir(parents=True, exist_ok=True)

        linked = 0
        missing = []
        for idx, row in enumerate(prompt_rows):
            case_name = row["name"]
            raw_mp4 = raw_by_case.get(case_name)
            if raw_mp4 is None:
                missing.append(case_name)
                continue
            dst = videos_dir / f"{idx + 1:04d}.mp4"
            if dst.exists() or dst.is_symlink():
                dst.unlink()
            dst.symlink_to(raw_mp4.resolve())
            linked += 1

        total_linked += linked
        note = f" ({len(missing)} not yet generated)" if missing else ""
        print(f"  {task}: linked {linked}/{len(prompt_rows)} -> {videos_dir}{note}")

    print(f"\nOrganized {total_linked} videos into {REVIDGEN_REPO / 'data' / MODEL_NAME}")

### 11.2 Run Five-Task QA Scoring (Qwen2.5-VL-72B, `qwen_local`)

Runs the ReVidgen scorer over each of the five task categories, then produces the per-category summary with `summary_scores.py`.


In [ ]:
%%bash
set -euo pipefail

if [ "${RUN_SCORING:-False}" != "True" ]; then
  echo "RUN_SCORING is not True. Set RUN_SCORING = True in cell 10.1 and re-run the prep cells first."
  exit 0
fi

cd "$REVIDGEN_REPO"
export HF_HOME="$RBENCH_SCORE_HF_HOME"
mkdir -p "$HF_HOME"
PY="$REVIDGEN_SCORER_ENV/bin/python"

for TASK in $RBENCH_SCORE_TASKS; do
  VIDEO_PATH="data/$MODEL_NAME/$TASK/videos"
  PROMPT_FILE="data/prompts/${TASK}_prompts.json"
  OUTPUT_PATH="results/5_tasks/$MODEL_NAME/$TASK/qwen_local"

  if [ ! -d "$VIDEO_PATH" ] || [ -z "$(ls -A "$VIDEO_PATH" 2>/dev/null)" ]; then
    echo "--- [skip] $TASK: no organized videos at $VIDEO_PATH ---"
    continue
  fi
  if [ ! -f "$PROMPT_FILE" ]; then
    echo "--- [skip] $TASK: prompt file missing: $PROMPT_FILE ---"
    continue
  fi

  echo "--- Scoring: $TASK ---"
  mkdir -p "$OUTPUT_PATH"
  CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" "$PY" eval/5_tasks/${TASK}.py \
    --model qwen_local \
    --video_path "$VIDEO_PATH" \
    --read_prompt_file "$PROMPT_FILE" \
    --output_path "$OUTPUT_PATH" \
    --qwen_model_path "$QWEN_MODEL_PATH" \
    --max_new_tokens 1024
  echo "--- Done: $TASK ---"
done

echo "--- Raw five-task scoring complete ---"
echo "Run Section 11.3 to validate all samples and compute the five-task score."

### 11.3 Validate and Compute the Five-Task Split Scores

Reads the unrounded per-sample results directly, requires exactly 50 valid scores for every task, accepts the supported action-score field names, normalizes each raw score with `(score - 1) / 4`, and averages the five task indicators. Section 13 reuses the same validated loader.


In [ ]:
import ast
import csv
import json
from statistics import fmean

RBENCH_ACTION_KEY_ALIASES = {
    "common_manipulation": ("action_execution", "action_effectiveness"),
    "multi-entity_collaboration": ("action_coordination", "action_effectiveness"),
}


def _parse_rbench_details(raw: str, *, task: str, sample: str) -> dict:
    for decoder in (json.loads, ast.literal_eval):
        try:
            details = decoder(raw)
        except (TypeError, ValueError, SyntaxError):
            continue
        if isinstance(details, dict):
            return details
    raise ValueError(f"{task}/{sample}: details are not a parseable object")


def _rbench_metric_score(details: dict, keys: tuple[str, ...], *, task: str, sample: str) -> float:
    present = [key for key in keys if key in details]
    if not present:
        raise KeyError(f"{task}/{sample}: missing {keys}; found {sorted(details)}")
    try:
        values = [float(details[key]["score"]) for key in present]
    except (KeyError, TypeError, ValueError) as error:
        raise ValueError(f"{task}/{sample}: invalid score under {present}") from error
    if any(not 1.0 <= value <= 5.0 for value in values):
        raise ValueError(f"{task}/{sample}: score outside 1-5: {values}")
    if any(value != values[0] for value in values[1:]):
        raise ValueError(f"{task}/{sample}: conflicting aliases {dict(zip(present, values))}")
    return values[0]


def load_rbench_task_scores(task: str) -> tuple[list[float], int]:
    result_csv = REVIDGEN_REPO / "results" / "5_tasks" / MODEL_NAME / task / "qwen_local" / "results.csv"
    if not result_csv.exists():
        raise FileNotFoundError(f"Missing five-task results: {result_csv}")
    with result_csv.open(newline="") as fp:
        rows = list(csv.DictReader(fp))
    if len(rows) != 50:
        raise ValueError(f"{task}: expected 50 result rows, found {len(rows)}")

    scores = []
    recovered = 0
    for row in rows:
        sample = row.get("name", "<unknown>")
        try:
            score = float(row.get("score", ""))
        except (TypeError, ValueError):
            score = -1.0

        if not 1.0 <= score <= 5.0:
            action_keys = RBENCH_ACTION_KEY_ALIASES.get(task)
            if action_keys is None:
                raise ValueError(f"{task}/{sample}: invalid raw score {row.get('score')!r}")
            details = _parse_rbench_details(row.get("details", ""), task=task, sample=sample)
            action = _rbench_metric_score(details, action_keys, task=task, sample=sample)
            completion = _rbench_metric_score(details, ("task_completion",), task=task, sample=sample)
            total = _rbench_metric_score(details, ("total",), task=task, sample=sample)
            score = max(min((action + completion) / 2.0, total), 1.0)
            recovered += 1
        scores.append(score)

    return scores, recovered


task_indicators = {}
print(f"RBench five-task split scores ({MODEL_NAME}):\n")
for task in RBENCH_SCORE_TASKS:
    raw_scores, recovered = load_rbench_task_scores(task)
    task_indicators[task] = fmean((score - 1.0) / 4.0 for score in raw_scores)
    suffix = f"  [recovered {recovered} key-mismatched rows]" if recovered else ""
    print(f"  {task:<28} {task_indicators[task]:.5f}{suffix}")

if len(task_indicators) != 5:
    raise AssertionError(f"Expected five task indicators, found {len(task_indicators)}")
five_task_score = sum(task_indicators.values()) / 5.0
print(f"\n  {'ALL_TASKS_MEAN':<28} {five_task_score:.5f}  ({five_task_score * 100:.2f}%)")


## 12. Score the Four Embodiments

This section evaluates `dual_arm`, `humanoid`, `single_arm`, and `quad` with all official ReVidgen components:

- Qwen VQA: robot/subject stability, physical plausibility, and task adherence/consistency.
- Motion operators: perceptible motion amplitude and Q-Align motion smoothness.
- Per-embodiment penalized aggregation and the final four-embodiment summary.

Set `RUN_EMBODIMENT_SCORING = True` in Section 10.1 and run the two-environment installation first.


### 12.1 Organize Four-Embodiment Videos

ReVidgen has incompatible filename expectations across its VQA and motion scripts. We create two symlink layouts without copying videos:

- VQA: numeric names such as `0001.mp4`.
- Motion metadata: benchmark names such as `dual_arm_0001.mp4`.


In [ ]:
if not RUN_EMBODIMENT_SCORING:
    print("RUN_EMBODIMENT_SCORING is False - enable it in Section 10.1 first.")
else:
    if not RBENCH_SCORE_RAW_DIR.exists():
        raise FileNotFoundError(f"Generated-video directory is missing: {RBENCH_SCORE_RAW_DIR}")

    raw_by_case: dict[str, Path] = {}
    for mp4 in sorted(RBENCH_SCORE_RAW_DIR.rglob("vision.mp4")):
        case_name = mp4.parent.name.split("__")[0]
        raw_by_case.setdefault(case_name, mp4.resolve())
    print(f"Found {len(raw_by_case)} generated clips under {RBENCH_SCORE_RAW_DIR}")

    for embodiment in RBENCH_EMBODIMENTS:
        prompt_path = REVIDGEN_REPO / "data" / "prompts" / f"{embodiment}_prompts.json"
        if not prompt_path.exists():
            raise FileNotFoundError(f"Missing scorer prompt file: {prompt_path}")
        prompt_rows = json.loads(prompt_path.read_text())

        vqa_dir = REVIDGEN_REPO / "data" / RBENCH_4EMB_VQA_NAME / embodiment / "videos"
        motion_dir = REVIDGEN_REPO / "data" / RBENCH_4EMB_MOTION_NAME / embodiment / "videos"
        vqa_dir.mkdir(parents=True, exist_ok=True)
        motion_dir.mkdir(parents=True, exist_ok=True)
        for videos_dir in (vqa_dir, motion_dir):
            for old_alias in videos_dir.glob("*.mp4"):
                if old_alias.is_symlink():
                    old_alias.unlink()

        linked = 0
        for position, row in enumerate(prompt_rows, start=1):
            case_name = row["name"]
            raw_mp4 = raw_by_case.get(case_name)
            if raw_mp4 is None:
                continue
            aliases = (vqa_dir / f"{position:04d}.mp4", motion_dir / f"{case_name}.mp4")
            for alias in aliases:
                if alias.exists():
                    raise FileExistsError(f"Refusing to replace a regular file: {alias}")
                if alias.is_symlink():
                    alias.unlink()
                alias.symlink_to(raw_mp4)
            linked += 1

        print(f"{embodiment}: linked {linked}/{len(prompt_rows)} videos")


### 12.2 Run the Three Four-Embodiment VQA Metrics

The Qwen jobs run sequentially with `.venv-rbench-scorer`.


In [ ]:
%%bash
set -euo pipefail

if [ "${RUN_EMBODIMENT_SCORING:-False}" != "True" ]; then
  echo "RUN_EMBODIMENT_SCORING is not True; skipping."
  exit 0
fi

cd "$REVIDGEN_REPO"
export HF_HOME="$RBENCH_SCORE_HF_HOME"
mkdir -p "$HF_HOME"
VLM_PY="$REVIDGEN_SCORER_ENV/bin/python"

for EMBODIMENT in $RBENCH_EMBODIMENTS; do
  VIDEO_PATH="data/$RBENCH_4EMB_VQA_NAME/$EMBODIMENT/videos"
  PROMPT_FILE="data/prompts/${EMBODIMENT}_prompts.json"
  if [ ! -d "$VIDEO_PATH" ] || [ -z "$(ls -A "$VIDEO_PATH" 2>/dev/null)" ]; then
    echo "--- [skip] $EMBODIMENT: no organized VQA videos ---"
    continue
  fi

  for METRIC in 1_robot_subject_stability 2_physical_plausibility 3_task_adherence_consistency; do
    OUTPUT_PATH="results/4_embodiments/$MODEL_NAME/$EMBODIMENT/VQA/qwen_local/$METRIC"
    mkdir -p "$OUTPUT_PATH"
    echo "--- VQA: $EMBODIMENT / $METRIC ---"
    CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" PYTHONNOUSERSITE=1 "$VLM_PY" \
      "eval/4_embodiments/$METRIC.py" \
      --model qwen_local \
      --video_path "$VIDEO_PATH" \
      --read_prompt_file "$PROMPT_FILE" \
      --output_path "$OUTPUT_PATH" \
      --qwen_model_path "$QWEN_MODEL_PATH" \
      --max_new_tokens 1024
  done
done


### 12.3 Run Motion Amplitude and Smoothness

Motion amplitude uses GroundingDINO, SAM2, and CoTracker. Smoothness uses Q-Align. All commands use `.venv-rbench-ops`.


In [ ]:
%%bash
set -euo pipefail
export MPLBACKEND=Agg

if [ "${RUN_EMBODIMENT_SCORING:-False}" != "True" ]; then
  echo "RUN_EMBODIMENT_SCORING is not True; skipping."
  exit 0
fi

cd "$REVIDGEN_REPO"
OPS_PY="$RBENCH_OPS_ENV/bin/python"
OPS_GPU="$RBENCH_OPS_CUDA_VISIBLE_DEVICES"

for EMBODIMENT in $RBENCH_EMBODIMENTS; do
  VIDEO_PATH="data/$RBENCH_4EMB_MOTION_NAME/$EMBODIMENT/videos"
  PROMPT_FILE="data/prompts/${EMBODIMENT}_prompts.json"
  MOTION_DIR="results/4_embodiments/$MODEL_NAME/$EMBODIMENT/motion"
  META_PATH="$MOTION_DIR/results.json"
  if [ ! -d "$VIDEO_PATH" ] || [ -z "$(ls -A "$VIDEO_PATH" 2>/dev/null)" ]; then
    echo "--- [skip] $EMBODIMENT: no organized motion videos ---"
    continue
  fi

  mkdir -p "$MOTION_DIR"
  PYTHONNOUSERSITE=1 "$OPS_PY" eval/4_embodiments/4_create_meta_info.py \
    -v "$VIDEO_PATH" -o "$META_PATH" -i "$PROMPT_FILE"
  CUDA_VISIBLE_DEVICES="$OPS_GPU" PYTHONNOUSERSITE=1 "$OPS_PY" \
    eval/4_embodiments/5_motion_amplitude.py \
    --meta_info_path "$META_PATH" \
    --target_type robotic_manipulator \
    --box_threshold 0.25 --text_threshold 0.20 --grid_size 30 --device cuda
done

for EMBODIMENT in $RBENCH_EMBODIMENTS; do
  META_PATH="results/4_embodiments/$MODEL_NAME/$EMBODIMENT/motion/results.json"
  [ -f "$META_PATH" ] || continue
  echo "--- Motion smoothness (serial): $EMBODIMENT ---"
  CUDA_VISIBLE_DEVICES="$OPS_GPU" PYTHONNOUSERSITE=1 "$OPS_PY" \
    eval/4_embodiments/6_motion_smoothness.py \
    --meta_info_path "$META_PATH" \
    --model_path checkpoints/q-future/one-align \
    --device cuda:0 --window_size 3
  PYTHONNOUSERSITE=1 "$OPS_PY" eval/4_embodiments/7_motion_total_score.py \
    --meta_info_path "$META_PATH"
done


### 12.4 Normalize ReVidgen Identifiers and Aggregate

Upstream ReVidgen emits `0001.jpeg`, `dual_arm_0001`, and a motion index `dual_arm_0001` for the same sample, while its aggregator accepts only digits. This idempotent cell canonicalizes generated result identifiers to `0001`; videos, prompts, and metric values are unchanged.


In [ ]:
import csv

if not RUN_EMBODIMENT_SCORING:
    print("RUN_EMBODIMENT_SCORING is False - nothing to normalize.")
else:
    result_root = REVIDGEN_REPO / "results" / "4_embodiments" / MODEL_NAME

    for embodiment in RBENCH_EMBODIMENTS:
        prefix = f"{embodiment}_"
        for metric in ("2_physical_plausibility", "3_task_adherence_consistency"):
            csv_path = result_root / embodiment / "VQA" / "qwen_local" / metric / "results.csv"
            if not csv_path.exists():
                continue
            with csv_path.open(newline="") as fp:
                reader = csv.DictReader(fp)
                fieldnames = reader.fieldnames or []
                rows = list(reader)
            for row in rows:
                if row.get("name", "").startswith(prefix):
                    row["name"] = row["name"][len(prefix):]
            with csv_path.open("w", newline="") as fp:
                writer = csv.DictWriter(fp, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(rows)

        motion_path = result_root / embodiment / "motion" / "results.json"
        if motion_path.exists():
            motion_rows = json.loads(motion_path.read_text())
            for row in motion_rows:
                if str(row.get("index", "")).startswith(prefix):
                    row["index"] = str(row["index"])[len(prefix):]
            motion_path.write_text(json.dumps(motion_rows, indent=4) + "\n")

        print(f"normalized result identifiers: {embodiment}")


In [ ]:
%%bash
set -euo pipefail

if [ "${RUN_EMBODIMENT_SCORING:-False}" != "True" ]; then
  echo "RUN_EMBODIMENT_SCORING is not True; skipping."
  exit 0
fi

cd "$REVIDGEN_REPO"
OPS_PY="$RBENCH_OPS_ENV/bin/python"

for EMBODIMENT in $RBENCH_EMBODIMENTS; do
  PYTHONNOUSERSITE=1 "$OPS_PY" eval/4_embodiments/8_summarize_robot_results.py \
    --i2v_model_name "$MODEL_NAME" \
    --robot_type "$EMBODIMENT" \
    --qwen_eval_name qwen_local
done

PYTHONNOUSERSITE=1 "$OPS_PY" eval/4_embodiments/summarize_i2v_results.py \
  --i2v_model_name "$MODEL_NAME"


### 12.5 Read the Four-Embodiment Split Scores

The final CSV contains one normalized, penalized row per embodiment plus `TOTAL_MEAN`.


In [ ]:
import csv

embodiment_summary = (
    REVIDGEN_REPO / "results" / "4_embodiments" / MODEL_NAME / "score_summary_qwen.csv"
)
if not embodiment_summary.exists():
    print(f"No summary found at {embodiment_summary}. Run Sections 12.1–12.4 first.")
else:
    with embodiment_summary.open(newline="") as fp:
        summary_rows = list(csv.DictReader(fp))
    required = ["PSS", "TAC", "RSS", "MS", "MA", "Task_Completion", "Visual_Quality"]
    for row in summary_rows:
        for field in required:
            if not row.get(field):
                raise ValueError(f"Incomplete four-embodiment summary: {row.get('Robot_Type')} / {field}")
            float(row[field])
    print(embodiment_summary)
    for row in summary_rows:
        print(row)


## 13. Compute the Official Overall RBench Score

The published RBench overall score gives equal weight to nine indicators:

```text
RBench overall = mean(5 task-category scores + 4 embodiment TS scores)
```

For each embodiment, `TS = (Task Completion + Visual Quality) / 2`. This cell reads the unrounded five-task result files and the penalized four-embodiment result files, requires the complete official sample counts (50 per task and 100 per embodiment), and writes the nine indicators plus their overall mean.


In [ ]:
import csv
from statistics import fmean

five_task_root = REVIDGEN_REPO / "results" / "5_tasks" / MODEL_NAME
embodiment_root = REVIDGEN_REPO / "results" / "4_embodiments" / MODEL_NAME
indicator_rows: list[dict] = []

# Compute five task indicators from unrounded per-video scores.
if "load_rbench_task_scores" not in globals():
    raise RuntimeError("Run Section 11.3 before computing the overall RBench score")
for task in RBENCH_SCORE_TASKS:
    task_scores, _ = load_rbench_task_scores(task)
    task_indicator = fmean((score - 1.0) / 4.0 for score in task_scores)
    indicator_rows.append({"split": "task", "indicator": task, "score": task_indicator})

# Compute each embodiment TS from its 100 per-video TC and VQ values.
for embodiment in RBENCH_EMBODIMENTS:
    result_csv = embodiment_root / embodiment / "score_summary_penalized_qwen.csv"
    if not result_csv.exists():
        raise FileNotFoundError(f"Missing embodiment results: {result_csv}")
    with result_csv.open(newline="") as fp:
        embodiment_rows = list(csv.DictReader(fp))
    sample_rows = [row for row in embodiment_rows if row.get("name") != "MEAN"]
    if len(sample_rows) != 100:
        raise ValueError(f"{embodiment}: expected 100 scored samples, found {len(sample_rows)}")
    sample_ts = []
    for row in sample_rows:
        try:
            tc = float(row["Task_Completion"])
            vq = float(row["Visual_Quality"])
        except (KeyError, TypeError, ValueError) as error:
            raise ValueError(f"{embodiment}: invalid TC/VQ row {row.get('name')}") from error
        sample_ts.append((tc + vq) / 2.0)
    embodiment_indicator = fmean(sample_ts)
    indicator_rows.append({"split": "embodiment", "indicator": embodiment, "score": embodiment_indicator})

if len(indicator_rows) != 9:
    raise AssertionError(f"Expected nine RBench indicators, found {len(indicator_rows)}")
if any(not 0.0 <= row["score"] <= 1.0 for row in indicator_rows):
    raise ValueError("At least one RBench indicator lies outside [0, 1]")

task_split_mean = fmean(row["score"] for row in indicator_rows if row["split"] == "task")
embodiment_split_mean = fmean(row["score"] for row in indicator_rows if row["split"] == "embodiment")
rbench_overall = fmean(row["score"] for row in indicator_rows)

output_rows = indicator_rows + [
    {"split": "aggregate", "indicator": "FIVE_TASKS_MEAN", "score": task_split_mean},
    {"split": "aggregate", "indicator": "FOUR_EMBODIMENTS_MEAN", "score": embodiment_split_mean},
    {"split": "aggregate", "indicator": "RBENCH_OVERALL", "score": rbench_overall},
]

output_dir = REVIDGEN_REPO / "results" / "rbench_overall" / MODEL_NAME
output_dir.mkdir(parents=True, exist_ok=True)
overall_csv = output_dir / "rbench_overall_qwen.csv"
with overall_csv.open("w", newline="") as fp:
    writer = csv.DictWriter(fp, fieldnames=["split", "indicator", "score"])
    writer.writeheader()
    writer.writerows(output_rows)

print(overall_csv)
for row in output_rows:
    print(f"{row['split']:<12} {row['indicator']:<30} {row['score']:.4f}")
print(f"\nRBench overall: {rbench_overall:.4f} ({rbench_overall * 100:.2f}%)")
